In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score
from scipy.stats.mstats import winsorize
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

/home/samuel/xai-health-risk-system/projectenv/lib/python3.12/site-packages/numpy/_core/getlimits.py:552: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


In [2]:
heart_disease_dataset = pd.read_csv("../data/processed/heart_processed.csv")

In [3]:
heart_disease_dataset.head()

,Age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [4]:
# Changing data types of categorical features
heart_disease_dataset['sex'] = heart_disease_dataset['sex'].astype('category')
heart_disease_dataset['fbs'] = heart_disease_dataset['fbs'].astype('category')
heart_disease_dataset['exang'] = heart_disease_dataset['exang'].astype('category')
heart_disease_dataset['target'] = heart_disease_dataset['target'].astype('category')
heart_disease_dataset['cp'] = heart_disease_dataset['cp'].astype('category')
heart_disease_dataset['restecg'] = heart_disease_dataset['restecg'].astype('category')
heart_disease_dataset['slope'] = heart_disease_dataset['slope'].astype('category')
heart_disease_dataset['thal'] = heart_disease_dataset['thal'].astype('category')
heart_disease_dataset['ca'] = heart_disease_dataset['ca'].astype('category')

In [5]:
heart_disease_dataset.corr()

,Age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
Age,1.000000,-0.103240,-0.071966,0.271121,0.219823,0.121243,-0.132696,-0.390227,0.088163,0.208137,-0.169105,0.271551,0.072297,-0.229324
sex,-0.103240,1.000000,-0.041119,-0.078974,-0.198258,0.027200,-0.055117,-0.049365,0.139157,0.084687,-0.026666,0.111729,0.198424,-0.279501
cp,-0.071966,-0.041119,1.000000,0.038177,-0.081641,0.079294,0.043581,0.306839,-0.401513,-0.174733,0.131633,-0.176206,-0.163341,0.434854
trestbps,0.271121,-0.078974,0.038177,1.000000,0.127977,0.181767,-0.123794,-0.039264,0.061197,0.187434,-0.120445,0.104554,0.059276,-0.138772
chol,0.219823,-0.198258,-0.081641,0.127977,1.000000,0.026917,-0.147410,-0.021772,0.067382,0.064880,-0.014248,0.074259,0.100244,-0.099966
fbs,0.121243,0.027200,0.079294,0.181767,0.026917,1.000000,-0.104051,-0.008866,0.049261,0.010859,-0.061902,0.137156,-0.042177,-0.041164
restecg,-0.132696,-0.055117,0.043581,-0.123794,-0.147410,-0.104051,1.000000,0.048411,-0.065606,-0.050114,0.086086,-0.078072,-0.020504,0.134468
thalach,-0.390227,-0.049365,0.306839,-0.039264,-0.021772,-0.008866,0.048411,1.000000,-0.380281,-0.349796,0.395308,-0.207888,-0.098068,0.422895
exang,0.088163,0.139157,-0.401513,0.061197,0.067382,0.049261,-0.065606,-0.380281,1.000000,0.310844,-0.267335,0.107849,0.197201,-0.438029
oldpeak,0.208137,0.084687,-0.174733,0.187434,0.064880,0.010859,-0.050114,-0.349796,0.310844,1.000000,-0.575189,0.221816,0.202672,-0.438441


In [6]:
# Winsorizing moderately skewed features
for col in ["trestbps"]:
    heart_disease_dataset[col] = winsorize(heart_disease_dataset[col])

In [7]:
#log-transforming highly skewed continous features
for col in ["oldpeak", "chol"]:
    heart_disease_dataset[col] = np.log1p(heart_disease_dataset[col])

In [8]:
#VIF Analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(df):
    vif = pd.DataFrame()
    vif['Features'] = df.columns
    vif['VIF'] = [variance_inflation_factor(df.values, i) for i in range(df.shape[1])]
    return vif

def drop_high_vif_features(df, threshold=5):
    dropped_features = []
    df = df.copy()
    while True:
        vif = calculate_vif(df)
        max_vif = vif["VIF"].max()
        if max_vif > threshold:
            drop_feature = vif.sort_values("VIF", ascending=False)["Features"].iloc[0]
            print(f"Dropping feature '{drop_feature}' with VIF '{max_vif:.3f}")
            df = df.drop(columns=[drop_feature])
            dropped_features.append(drop_feature)
            print("Remaining features\n", df.columns.tolist())
        else:
            break
    return df, dropped_features
X = heart_disease_dataset.drop(columns=['target'])
X_reduced, dropped = drop_high_vif_features(X, threshold=10.0)
print("Dropped features:", dropped)
print("Final VIFs:\n", calculate_vif(X_reduced))

Dropping feature 'chol' with VIF '186.220
Remaining features
 ['Age', 'sex', 'cp', 'trestbps', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Dropping feature 'trestbps' with VIF '56.819
Remaining features
 ['Age', 'sex', 'cp', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Dropping feature 'thalach' with VIF '29.094
Remaining features
 ['Age', 'sex', 'cp', 'fbs', 'restecg', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Dropping feature 'Age' with VIF '18.797
Remaining features
 ['sex', 'cp', 'fbs', 'restecg', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
Dropping feature 'thal' with VIF '10.729
Remaining features
 ['sex', 'cp', 'fbs', 'restecg', 'exang', 'oldpeak', 'slope', 'ca']
Dropped features: ['chol', 'trestbps', 'thalach', 'Age', 'thal']
Final VIFs:
   Features       VIF
0      sex  3.153505
1       cp  2.102299
2      fbs  1.223841
3  restecg  1.962503
4    exang  1.905551
5  oldpeak  2.428587
6    slope  3.681167
7       ca  1.696329

In [9]:
# Dividing dataset into exploratory features and target
X = heart_disease_dataset.drop("target", axis=1)
y = heart_disease_dataset["target"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

In [12]:
# Performing GridSeearchCV for ElasticNet Model
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

elasticnet_lr = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, random_state=42)
grid_search = GridSearchCV(elasticnet_lr, param_grid, cv=5, scoring='recall')
grid_search.fit(X_train_norm, y_train)

print("Best parameters:", grid_search.best_params_)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_norm)

Best parameters: {'C': 0.01, 'l1_ratio': 0.1}


In [13]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'saga']  # Optional: tune solver
}

lasso_lr = LogisticRegression(penalty='l1', max_iter=5000, random_state=42)
grid_search_lasso = GridSearchCV(lasso_lr, param_grid, cv=5, scoring='recall')
grid_search_lasso.fit(X_train_norm, y_train)

print("Best parameters:", grid_search_lasso.best_params_)
best_lasso = grid_search_lasso.best_estimator_
y_pred_lasso = best_lasso.predict(X_test_norm)

Best parameters: {'C': 0.1, 'solver': 'saga'}


In [14]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['lbfgs', 'saga']  # Both support L2 penalty
}

ridge_lr = LogisticRegression(penalty='l2', max_iter=5000, random_state=42)
grid_search_ridge = GridSearchCV(ridge_lr, param_grid, cv=5, scoring='recall')
grid_search_ridge.fit(X_train_norm, y_train)

print("Best parameters:", grid_search_ridge.best_params_)
best_ridge = grid_search_ridge.best_estimator_
y_pred_ridge = best_ridge.predict(X_test_norm)

Best parameters: {'C': 0.01, 'solver': 'lbfgs'}


In [15]:
# GridSearchCV for Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', None]
}

rf = RandomForestClassifier(random_state=42)
grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='recall')
grid_search_rf.fit(X_train_norm, y_train)

print("Best parameters for Random Forest:", grid_search_rf.best_params_)
best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test_norm)

Best parameters for Random Forest: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
